# Setup

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import pickle as pkl
import time
from scipy.stats import ttest_1samp
%matplotlib inline

import matplotlib.font_manager as fm
fm.fontManager.addfont('/work/magroup/skrieger/Arial.ttf')
matplotlib.rcParams['font.size'] = 12.0
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['svg.fonttype'] = 'none'

%load_ext autoreload
%autoreload 2

# Eykthyr imports
import sys
sys.path.insert(0, '.')
from eykthyr.eykthyr import Eykthyr, load_anndata
import eykthyr.plotting as epl
from eykthyr.pseudotime import Gradient_calculator

In [ ]:
# Load the pre-computed Eykthyr object
# Adjust the path to where the Eykthyr object was saved via e.save_anndata()
ey = load_anndata('../eykthyrspatialatacrna.h5ad')

# Convenience aliases
datasets = ey.perturbed_X   # list of perturbed AnnData (one per sample)
samples = ey.datasetnames
print(samples)

# Panel A

In [ ]:
leiden_labels = {'0': 'Muscle',
                 '1': 'Cartilage',
                 '2': 'Cartilage',
                 '3': 'Hindbrain',
                 '4': 'Muscle',
                 '5': 'Forebrain',
                 '6': 'CNS',
                 '7': 'Ventricle',
                 '8': 'Eye',
                 '9': 'Ganglia'
                }

datasets[0].obs['labeled_leiden'] = [leiden_labels[c] for c in datasets[0].obs['original_leiden']]

# Rotate/flip spatial coordinates for better visualization
# (original: x -> 50-y, y -> x)
ad = datasets[0].copy()
ad.obsm['spatial_2'] = ad.obsm['spatial'].copy()
ad.obsm['spatial'][:,0] = 50 - datasets[0].obsm['spatial'][:,1]
ad.obsm['spatial'][:,1] = datasets[0].obsm['spatial'][:,0]
datasets[0] = ad
leiden_colors = {
    'Ventricle': '#377eb8',
    'Hindbrain': '#e41a1c',
    'Cartilage': '#984ea3',
    'Muscle': '#f781bf',
    'CNS': '#4daf4a',
    'Eye': '#ffff33',
    'Ganglia': '#a65628',
    'Forebrain': '#ff7f00'}
for d in datasets:
    d.uns['labeled_leiden_colors'] = [leiden_colors[a] for a in sorted(d.obs['labeled_leiden'].unique().astype('str'))]

In [ ]:
# This plots panel A
sc.pl.spatial(datasets[0], color='labeled_leiden', spot_size=1, frameon=False, title='', save='Panel_3A.svg')

# Supp Panels A & B — Spatial Pseudotime

In [ ]:
# Compute spatial distance to Ventricle cells as pseudotime proxy
sc.pp.neighbors(datasets[0], use_rep='spatial', key_added='spatial_neighbors')
nns = sc.Neighbors(datasets[0], neighbors_key='spatial_neighbors')
nns.compute_neighbors(knn=False, use_rep='spatial', method='gauss')

ventricle_cells = datasets[0][datasets[0].obs['labeled_leiden'] == 'Ventricle']
datasets[0].obs['ventricle_distance'] = nns.distances[:, datasets[0].obs['labeled_leiden'] == 'Ventricle'].min(axis=1)
epl.prep_paga(ey, 'original_leiden')
for d in datasets:
    d.obsm['spatial_2'] = d.obsm['spatial'].astype(np.float64).copy()
    d.obsm['spatial_2'][:,1] = d.obsm['spatial_2'][:,1] * -1

exadatas = ey.RNA
exadatas[0].obsm['spatial'] = datasets[0].obsm['spatial']
tfadata = ey.TF[0]
tfadata.obsm['spatial'] = datasets[0].obsm['spatial']

In [ ]:
# This plots Supp panel A
subset_X = [datasets[0][(datasets[0].obs['labeled_leiden'] == 'Hindbrain') |
                         (datasets[0].obs['labeled_leiden'] == 'Ventricle')].copy()]

# Further spatial filtering
subset_X2 = subset_X[0][subset_X[0].obsm['spatial'][:,0] < 25].copy()
subset_X3_ad = subset_X2[subset_X2.obsm['spatial'][:,1] > 20].copy()

sc.pl.spatial(subset_X3_ad, spot_size=1, color='ventricle_distance', title='', cmap='magma',
              frameon=False, save='Panel_supp1.svg')

In [ ]:
# This plots Supp panel B
sc.pp.neighbors(subset_X3_ad, use_rep='X')
sc.tl.paga(subset_X3_ad, groups='labeled_leiden')
subset_X3_ad.uns['iroot'] = 323
sc.tl.dpt(subset_X3_ad)
sc.pl.spatial(subset_X3_ad, spot_size=1, color='dpt_pseudotime', title='', cmap='magma',
              frameon=False, save='Panel_supp2.svg')

# Panels B, C, D, H, I — TF Perturbation Simulation

In [ ]:
# Create a subset Eykthyr object for the Hindbrain+Ventricle region
subset_X3 = [subset_X3_ad]

leiden_colors_sub = {'Ventricle': '#377eb8', 'Hindbrain': '#e41a1c'}
for d in subset_X3:
    d.uns['labeled_leiden_colors'] = [leiden_colors_sub[a] for a in sorted(d.obs['labeled_leiden'].unique().astype('str'))]

ey_subset = Eykthyr()
ey_subset.perturbed_X = [subset_X3_ad]
ey_subset.num_metagenes = ey.num_metagenes
# Subset TF activity data to match subset_X3 cells
tfadata_sub = tfadata[subset_X3[0].obs_names].copy()
tfadata_sub.obs['ventricle_distance'] = subset_X3[0].obs['ventricle_distance']
tfadata_sub.obs['dpt_pseudotime'] = subset_X3[0].obs['dpt_pseudotime']
tfadata_sub.obs['labeled_leiden'] = subset_X3[0].obs['labeled_leiden']
tfadata_sub.obsm['spatial_2'] = subset_X3[0].obsm['spatial_2']

# Copy TF activity scores into obs for gradient analysis
X = tfadata_sub.X.toarray() if hasattr(tfadata_sub.X, 'toarray') else tfadata_sub.X
tf_scores = pd.DataFrame(X, index=tfadata_sub.obs_names, columns=tfadata_sub.var_names)
tfadata_sub.obs = pd.concat([tfadata_sub.obs, tf_scores], axis=1)

In [ ]:
# This plots panels B, C, D, H, and I
# File mapping:
#   {fig_prefix}_differentiation.svg        -> Panel B (reference flow, spatial)
#   {fig_prefix}_Hes1_inner_product_abs.svg  -> Panels C (observed PS) and D (random PS)
#   {fig_prefix}_Hes1_inner_product.svg      -> Panel H (PS + arrows, spatial)
#   umap_spatial_simulation (X_draw_graph_fr) -> Panel I
ey_subset.perturbed_X[0].obs['ventricle_distance_backup'] = ey_subset.perturbed_X[0].obs.get('ventricle_distance', np.nan)
ey_subset.perturbed_X[0].obs['ventricle_distance'] = subset_X3_ad.obs['dpt_pseudotime']

ips_single = epl.development_simulation(
    ey_subset, ['Hes1'],
    n_grid=20,
    min_masses=[1, 0.008],
    scales=[12, 0.3],
    embeddings=['spatial_2', 'X_draw_graph_fr'],
    n_neighbors=[20, 25],
    show_plots=[True, True],
    vm=0.6,
    arrow_args={'width': 0.012, 'headwidth': 2.5, 'headlength': 2.5, 'headaxislength': 2},
    save_figs=True,
    fig_prefix='Fig3',
)


# Panels E & F, Supp Panel C — TF Ranking

In [ ]:
all_tfs = [t.split('_')[1] for t in ey.perturbed_X[0].obsm_keys() if 'dropout' in t and 'normalized' not in t]

In [ ]:
# Score all TFs by pseudotime-alignment inner product (dpt_pseudotime)
ips = []
for tf in all_tfs:
    result = epl.development_simulation(
        ey_subset, [tf],
        n_grid=20,
        min_masses=[1],
        scales=[12],
        embeddings=['spatial_2'],
        n_neighbors=[20],
        show_plots=False,
        vm=0.6,
    )
    ips.extend(result)

pkl.dump(ips, open('ips_pseudotime.pkl','wb'))

In [ ]:
ips = pkl.load(open('ips_pseudotime.pkl','rb'))

ips.sort(key=lambda tup: tup[1], reverse=True)

lit_tfs = ['Pax6', 'Emx1', 'Gsh1', 'Gsh2', 'Er81', 'Sp8', 'Nkx2.1', 'Dlx1', 'Dlx2', 'Olig2',
           'Ngn2', 'Mash1', 'Gsx1', 'Gsx2', 'Etv1', 'Nkx21', 'Neurog2', 'Ascl1']
available_lit_tfs = [tf for tf in lit_tfs if tf in all_tfs]
ips_tfs = [tup[0] for tup in ips]
available_lit_tf_ranks = [ips_tfs.index(tf) for tf in available_lit_tfs]
from scipy.stats import ttest_1samp
available_lit_tf_ranks_devs = available_lit_tf_ranks
n_tfs_devs = len(ips_tfs)
# Restore ventricle_distance if overwritten
if 'ventricle_distance_backup' in subset_X3_ad.obs.columns:
    subset_X3_ad.obs['ventricle_distance'] = subset_X3_ad.obs['ventricle_distance_backup']
    ey_subset.perturbed_X[0].obs['ventricle_distance'] = subset_X3_ad.obs['ventricle_distance']

In [ ]:
# Score all TFs by ventricle_distance-alignment inner product
# Load from cache if available
# ips2 = pkl.load(open('ips_2.pkl','rb'))
ips2 = []
for tf in all_tfs:
    result = epl.development_simulation(
        ey_subset, [tf],
        n_grid=20,
        min_masses=[1],
        scales=[12],
        embeddings=['spatial_2'],
        n_neighbors=[20],
        show_plots=[False],
        vm=0.6,
    )
    ips2.extend(result)
pkl.dump(ips2, open('ips_2.pkl','wb'))

In [ ]:
ips2 = pkl.load(open('ips_2.pkl','rb'))
ips2 = [(p[0], abs(p[1])) for p in ips2]
ips2.sort(key=lambda tup: tup[1], reverse=True)
lit_tfs = ['Pax6', 'Emx1', 'Gsh1', 'Gsh2', 'Er81', 'Sp8', 'Nkx2.1', 'Dlx1', 'Dlx2', 'Olig2',
           'Ngn2', 'Mash1', 'Gsx1', 'Gsx2', 'Etv1', 'Nkx21', 'Neurog2', 'Ascl1']
available_lit_tfs = [tf for tf in lit_tfs if tf in all_tfs]
ips_tfs = [tup[0] for tup in ips2]
available_lit_tf_ranks = [ips_tfs.index(tf) for tf in available_lit_tfs]

In [ ]:
# This plots panel F
bestrankss2 = available_lit_tf_ranks
bestcols = ['Score']
fig, ax = plt.subplots(1,1,figsize=(5,2))
bp = ax.boxplot(bestrankss2, patch_artist=True, labels=bestcols, vert=False)
plt.setp(bp['boxes'], color='#1c9099')
plt.setp(bp['medians'], color='red')
plt.setp(bp['whiskers'], color='black')
plt.setp(bp['fliers'], color='black')
for patch in bp['boxes']:
    patch.set_edgecolor('black')
ax.set(title="Eykthyr ranking known TFs")
ax.axvline(len(ips_tfs) / 2, linestyle='--', color='black')
plt.savefig(f'ventricle_tf_scores_horizontal.svg', bbox_inches='tight')

In [ ]:
# This plots panel E
X = ips_tfs[:20]
height = [tup[1] for tup in ips2[:20]]
X.reverse()
height.reverse()
matplotlib.rcParams['font.size'] = 16.0
fig, ax = plt.subplots(1,1,figsize=(5,2))
markerline, stemlines, baseline = ax.stem(X, height, orientation='vertical', bottom=25,
                                          linefmt='#1c9099', markerfmt='C0o', basefmt='w-')
ax.set_ylabel('score')
ax.tick_params(axis='x', labelrotation=60)
markerline.set_markerfacecolor('#1c9099')
markerline.set_markeredgecolor('#1c9099')
markerline.set_markersize(8)
markerline.set_color('#1c9099')
plt.savefig(f'ventricle_top_tfs_horizontal.svg', bbox_inches='tight')

In [ ]:
ips = pkl.load(open('correlation_based_tf_ranking_pseudotime.pkl','rb'))
ips = ips.sort_values(by='score_inner_product', ascending=False)
lit_tfs = ['Pax6', 'Emx1', 'Gsh1', 'Gsh2', 'Er81', 'Sp8', 'Nkx2.1', 'Dlx1', 'Dlx2', 'Olig2',
           'Ngn2', 'Mash1', 'Gsx1', 'Gsx2', 'Etv1', 'Nkx21', 'Neurog2', 'Ascl1']
available_lit_tfs = [tf for tf in lit_tfs if tf in all_tfs]
ips_tfs = ips['TF'].values.tolist()
available_lit_tf_ranks = [ips_tfs.index(tf) for tf in available_lit_tfs]
available_lit_tf_ranks
available_lit_tf_ranks_corr = available_lit_tf_ranks
n_tfs_corr = len(ips_tfs)

In [ ]:
# This plots Supp panel C
# Combines development_simulation ranking (Eykthyr) and correlation-based ranking (Correlation)
n_tfs = n_tfs_devs  # same total for both methods
fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
bp = ax.boxplot(
    [available_lit_tf_ranks_devs, available_lit_tf_ranks_corr],
    patch_artist=True,
    labels=['Eykthyr', 'Correlation'],
    vert=False,
)
colors = ['#1c9099', '#de8b3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('black')
plt.setp(bp['medians'], color='red')
plt.setp(bp['whiskers'], color='black')
plt.setp(bp['fliers'], color='black')
ax.set_xlim(0, n_tfs)
ax.axvline(n_tfs / 2, linestyle='--', color='black')
ax.set_xlabel('Rank')
ax.set_title('Known TF ranking')
plt.savefig('ventricle_tf_scores_supp_panel_C.svg', bbox_inches='tight')

# Panel G — Metagene Expression

In [ ]:
# This plots panel G (top)
for x in [13]:
    datasets[0].obs[f'X_{x}'] = datasets[0].obsm['X'][:,x]
    sc.pl.spatial(datasets[0], color=f'X_{x}', spot_size=1, frameon=False,
                  title=f'Metagene {x} expression', cmap='viridis', colorbar_loc=None,
                  save=f'm{x}_expression.svg')

In [ ]:
# This plots panel G (bottom)
tf = 'Hes1'
for x in [13]:
    datasets[0].obs[f'{tf}_X_{x}'] = datasets[0].obsm[f'X_{tf}_dropout'][:,x] - datasets[0].obsm['X'][:,x]
    sc.pl.spatial(datasets[0], color=f'{tf}_X_{x}', spot_size=1, cmap='bwr', vmin=-0.08, vmax=0.08,
                  frameon=False, title=f'{tf} M{x} delta', colorbar_loc=None,
                  save=f'{tf}_{x}_delta.svg')

# Panel J — Gene Response to Hes1 Perturbation

In [ ]:
# This plots panel J
import gseapy as gp
from gseapy import barplot, dotplot

def get_response_dfs(tf, datasets, datasetnames):
    """
    Compute per-cell-type gene-level response to TF perturbation.
    Uses datasets[i].uns['M'][datasetname] metagene-to-gene matrix.
    """
    response_dfs = [pd.DataFrame(index=d.var.index, columns=d.obs['labeled_leiden'].unique())
                    for d in datasets]
    for d, df, dname in zip(datasets, response_dfs, datasetnames):
        d.obsm[f'{tf}_mse'] = d.obsm['X'] - d.obsm[f'X_{tf}_dropout']
        K = d.obsm['X'].shape[1]
        ct_df = pd.DataFrame(index=[f'X_{i}' for i in range(K)],
                             columns=[ct for ct in d.obs['labeled_leiden'].unique()])
        for ct in d.obs['labeled_leiden'].unique():
            ct_mse = d[d.obs['labeled_leiden'] == ct].obsm[f'{tf}_mse'].mean(axis=0)
            ct_df.loc[:, ct] = ct_mse
            gene_ct_mse = np.matmul(d.uns['M'][dname], ct_mse)
            df.loc[:, ct] = gene_ct_mse
        absmax = max(abs(ct_df.to_numpy().min()), abs(ct_df.to_numpy().max()))
        fig, ax = plt.subplots(1,1,figsize=(8,8))
        ax.set_yticks(np.arange(len(ct_df.columns)), labels=ct_df.columns)
        ax.set_xticks(np.arange(len(ct_df.index)), labels=[f'{k}' for k in range(len(ct_df.index))])
        ax.imshow(ct_df.to_numpy().astype(float).T, cmap='bwr', vmin=-absmax, vmax=absmax)
        plt.savefig(f'{tf}_mchange.svg', bbox_inches='tight')
        plt.show()
    return response_dfs

ans = get_response_dfs('Hes1', [datasets[0]], ['spatialatacrna_mouseembryo2'])

In [ ]:
hesgenes = []
for line in open('../hes_genes.csv','r').readlines():
    hesgenes.append(line.strip())
possiblegenes = []
for line in open('../possible_genes.csv','r').readlines():
    possiblegenes.append(line.strip())

possiblesortedind = np.asarray(ans[0][ans[0].index.isin(possiblegenes)].sort_values(
    by='Ventricle', ascending=True).index).astype(str)
l2 = []
for g in hesgenes:
    lil = np.where(possiblesortedind == g)
    if len(lil[0]) > 0:
        l2.append(lil[0][0])

In [ ]:
# This plots supp panel A (different Supp figure)
from sklearn.metrics import precision_recall_curve, average_precision_score

y_true = [1 if gene in hesgenes else 0 for gene in possiblesortedind]
y_scores = [-1 * r for r in range(len(possiblesortedind))]

precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = average_precision_score(y_true, y_scores)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.plot(recall, precision, color='#1c9099', linewidth=2, label=f'AUPRC = {auprc:.2f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve for Known TFs')
ax.legend(loc='lower left')
ax.grid(True)
plt.savefig(f'Eykthyr_PR_curve_Hes1.svg', bbox_inches='tight')

# Panel K — GSEA

In [ ]:
def do_gsea(glist, cell_type, background, get_dotplot=True):
    enr = gp.enrichr(gene_list=glist,
                 gene_sets=['GO_Biological_Process_2023','Reactome_2022'],
                 organism='mouse',
                 outdir=None,
                )
    enr.results.sort_values(by='Adjusted P-value')
    if get_dotplot == True:
        ax = dotplot(enr.results,
              column="Adjusted P-value",
              x='Gene_set',
              size=10,
              top_term=5,
              figsize=(3,5),
              title=f"GSEA {cell_type}",
              xticklabels_rot=45,
              show_ring=True,
              marker='o',
              ofname=f'{cell_type}_dotplot.svg',
             )
    ax2 = barplot(enr.results,
              column="Adjusted P-value",
              group='Gene_set',
              size=10,
              top_term=5,
              figsize=(3,5),
              color=['red','green','blue'],
              title=f'GSEA {cell_type}',
              ofname=f'{cell_type}_barplot.svg',
             )
    return enr

In [ ]:
ans = get_response_dfs('Hes1', [datasets[0]], ['spatialatacrna_mouseembryo2'])
topk = []
k = 200
bigk = k * len(ans[0].columns)
for column in ans[0].columns:
    topkrows = ans[0].sort_values(by=column, ascending=True).index[:bigk]
    for row in topkrows:
        topk.append((ans[0].loc[row,column], column, row))
topk.sort()
topkchanged = topk[:bigk]
glists = [[g[2] for g in topkchanged if g[1] == ct] for ct in ans[0].columns]
enrs = []
cts_done = []
for i, ct in enumerate(ans[0].columns):
    if len(glists[i]) > k:
        time.sleep(100)
        enrs.append(do_gsea(glists[i][:k], ct, datasets[0].var_names.to_list(), get_dotplot=True))
        cts_done.append(ct)

In [ ]:
dpallmenr = enrs[0]
GOenr = dpallmenr.results[dpallmenr.results['Gene_set'] == 'GO_Biological_Process_2023'].sort_values(by='Adjusted P-value')
reactomeenr = dpallmenr.results[dpallmenr.results['Gene_set'] == 'Reactome_2022'].sort_values(by='Adjusted P-value')
X1 = GOenr['Term'].values[:5].tolist()
X2 = reactomeenr['Term'].values[:5].tolist()
X1 = [' '.join(x.split()[:-1]) for x in X1]
X2 = [' '.join(x.split()[:-1]) for x in X2]
X1[0] = f'{X1[0]} (GO)'
X1[1] = f'{X1[1]} (GO)'
X1.reverse()
X2.reverse()
height1 = GOenr['Adjusted P-value'].values[:5].tolist()
height2 = reactomeenr['Adjusted P-value'].values[:5].tolist()
height1 = [np.log10(1 / float(h)) for h in height1]
height2 = [np.log10(1 / float(h)) for h in height2]
height1.reverse()
height2.reverse()
marker_sizes1 = GOenr['Odds Ratio'].values[:5].tolist()
marker_sizes2 = reactomeenr['Odds Ratio'].values[:5].tolist()
marker_sizes1.reverse()
marker_sizes2.reverse()
desired_max = 20
desired_min = 5
min_marker_size = min(min(marker_sizes1), min(marker_sizes2))
max_marker_size = max(max(marker_sizes1), max(marker_sizes2))
marker_sizes1 = [(m - min_marker_size) + desired_min for m in marker_sizes1]
marker_sizes2 = [(m - min_marker_size) + desired_min for m in marker_sizes2]
max_marker_size = max(max(marker_sizes1), max(marker_sizes2))
marker_sizes1 = [(m / max_marker_size) * (desired_max - desired_min) for m in marker_sizes1]
marker_sizes2 = [(m / max_marker_size) * (desired_max - desired_min) for m in marker_sizes2]
marker_sizes1 = [m + desired_min for m in marker_sizes1]
marker_sizes2 = [m + desired_min for m in marker_sizes2]

In [ ]:
# This plots panel K
tf = 'Hes1'
ct = cts_done[0]
fig, ax = plt.subplots(1,1,figsize=(3,5))
for ms, x, h in zip(marker_sizes2, X2, height2):
    markerline2, stemlines2, baseline2 = ax.stem(x, h, orientation='horizontal', markerfmt='C0o', basefmt='w-',
                                             label='Reactome', linefmt='#7570b3')
    markerline2.set_markerfacecolor('#7570b3')
    markerline2.set_markeredgecolor('#7570b3')
    markerline2.set_markersize(ms)
for ms, x, h in zip(marker_sizes1, X1, height1):
    markerline, stemlines, baseline = ax.stem(x, h, orientation='horizontal', markerfmt='C0o', basefmt='w-',
                                          label='GO', linefmt='#1b9e77')
    markerline.set_markerfacecolor('#1b9e77')
    markerline.set_markeredgecolor('#1b9e77')
    markerline.set_markersize(ms)
ax.set_xlabel(r'$-\log$(adj. p-val.)')
plt.savefig(f'{tf}_{ct}_GSEA.svg', bbox_inches='tight')